In [22]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_rel

from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    BaggingRegressor,
    GradientBoostingRegressor,
)

In [23]:
# ============================================================
# Paper-style solar forecasting reproduction (revised)
# Based on:
# "Solar Energy Forecasting Using Machine Learning Techniques
#  for Enhanced Grid Stability"
#
# This version reflects the user's requested corrections:
# - 9 comparison models
# - 4 evaluation metrics (RMSE, MSE, MAE, R2)
# - 5-fold CV + grid search for GBR
# - detailed comparison among SVR / RF / GBR
# - paired t-tests on fold RMSE values
# - 23 requested tables/figures
#
# IMPORTANT
# - Exact numeric reproduction of the paper is not guaranteed because
#   the original paper does not disclose every split/preprocessing detail.
# - The paper text supports GBR grid search with:
#   learning_rate in {0.05, 0.10}, max_depth in {3, 5},
#   n_estimators in {100, 150}, subsample in {0.8, 1.0}.
# ============================================================

In [24]:
# -----------------------------
# 0) Paths
# -----------------------------
DATA_PATH = Path('spg.csv')
OUT_DIR = Path('paper_reproduction_outputs_revised2')
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [25]:
# -----------------------------
# 1) Load data
# -----------------------------
df = pd.read_csv(DATA_PATH)

print('=' * 90)
print('Loaded dataset')
print(f'Shape: {df.shape}')
print('Columns:')
for c in df.columns:
    print(' -', c)
print('=' * 90)

Loaded dataset
Shape: (4213, 21)
Columns:
 - temperature_2_m_above_gnd
 - relative_humidity_2_m_above_gnd
 - mean_sea_level_pressure_MSL
 - total_precipitation_sfc
 - snowfall_amount_sfc
 - total_cloud_cover_sfc
 - high_cloud_cover_high_cld_lay
 - medium_cloud_cover_mid_cld_lay
 - low_cloud_cover_low_cld_lay
 - shortwave_radiation_backwards_sfc
 - wind_speed_10_m_above_gnd
 - wind_direction_10_m_above_gnd
 - wind_speed_80_m_above_gnd
 - wind_direction_80_m_above_gnd
 - wind_speed_900_mb
 - wind_direction_900_mb
 - wind_gust_10_m_above_gnd
 - angle_of_incidence
 - zenith
 - azimuth
 - generated_power_kw


In [30]:
# -----------------------------
# 2) Target and features
# -----------------------------
TARGET = 'generated_power_kw'

FEATURES = [
    'temperature_2_m_above_gnd',
    'relative_humidity_2_m_above_gnd',
    'mean_sea_level_pressure_MSL',
    'total_precipitation_sfc',
    'snowfall_amount_sfc',
    'total_cloud_cover_sfc',
    'high_cloud_cover_high_cld_lay',
    'medium_cloud_cover_mid_cld_lay',
    'low_cloud_cover_low_cld_lay',
    'shortwave_radiation_backwards_sfc',
    'wind_speed_10_m_above_gnd',
    'wind_direction_10_m_above_gnd',
    'wind_speed_80_m_above_gnd',
    'wind_direction_80_m_above_gnd',
    'wind_speed_900_mb',
    'wind_direction_900_mb',
    'wind_gust_10_m_above_gnd',
    'angle_of_incidence',
    'zenith',
    'azimuth',
]

core_features_v1 = [
    "angle_of_incidence",
    "azimuth",
    "zenith",
    "shortwave_radiation_backwards_sfc",
    "total_cloud_cover_sfc",
    "mean_sea_level_pressure_MSL",
    "relative_humidity_2_m_above_gnd",
    "temperature_2_m_above_gnd",
    "wind_gust_10_m_above_gnd",
    "high_cloud_cover_high_cld_lay"
]

missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

X = df[FEATURES].copy()
y = df[TARGET].copy()

In [31]:
# -----------------------------
# 3) Split and CV
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=False)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

Train shape: (3370, 20), Test shape: (843, 20)


In [33]:
# -----------------------------
# 4) Preprocessors
# -----------------------------
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            FEATURES,
        )
    ],
    remainder='drop',
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            FEATURES,
        )
    ],
    remainder='drop',
)

In [36]:
# -----------------------------
# 5) Helpers
# -----------------------------
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
    }


def evaluate_test(model, X_train, y_train, X_test, y_test):
    est = clone(model)
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    return est, pred, regression_metrics(y_test, pred)


def save_table(df_table, filename):
    path = OUT_DIR / filename
    df_table.to_csv(path, index=False)
    return path


def plot_bar(x, y, title, xlabel, ylabel, filename, rotation=0):
    plt.figure(figsize=(10, 6))
    plt.bar(x, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if rotation:
        plt.xticks(rotation=rotation, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=200)
    plt.close()


def build_fold_rmse_table(model_dict, X_data, y_data, cv_obj):
    table = {'Fold': [f'Fold {i}' for i in range(1, cv_obj.get_n_splits() + 1)]}
    for name, model in model_dict.items():
        fold_scores = []
        for train_idx, valid_idx in cv_obj.split(X_data, y_data):
            X_tr = X_data.iloc[train_idx]
            X_va = X_data.iloc[valid_idx]
            y_tr = y_data.iloc[train_idx]
            y_va = y_data.iloc[valid_idx]
            est = clone(model)
            est.fit(X_tr, y_tr)
            pred = est.predict(X_va)
            rmse = np.sqrt(mean_squared_error(y_va, pred))
            fold_scores.append(rmse)
        table[name] = fold_scores
    df_fold = pd.DataFrame(table)
    mean_row = {'Fold': 'Mean'}
    for name in model_dict:
        mean_row[name] = df_fold[name].mean()
    df_fold = pd.concat([df_fold, pd.DataFrame([mean_row])], ignore_index=True)
    return df_fold


def fit_line(x, y):
    coeffs = np.polyfit(np.asarray(x), np.asarray(y), deg=1)
    return coeffs

In [38]:
# -----------------------------
# 6) Models (9 total)
# -----------------------------
models = {
    'Multiple Linear Regression': Pipeline([
        ('prep', scaled_preprocessor),
        ('model', LinearRegression()),
    ]),
    'Ridge Regression': Pipeline([
        ('prep', scaled_preprocessor),
        ('model', Ridge(alpha=80.0)),
    ]),
    'LASSO Regression': Pipeline([
        ('prep', scaled_preprocessor),
        ('model', Lasso(alpha=3.5, max_iter=10000)),
    ]),
    'Decision Tree Regression': Pipeline([
        ('prep', tree_preprocessor),
        ('model', DecisionTreeRegressor(
            max_depth=8,
            min_samples_split=2,
            min_samples_leaf=17,
            random_state=42
        )),
    ]),
    'Support Vector Regression': Pipeline([
        ('prep', scaled_preprocessor),
        ('model', SVR(
            kernel='rbf',
            C=2.5,
            epsilon=0.1,
            gamma='scale'
        )),
    ]),
    'Random Forest Regression': Pipeline([
        ('prep', tree_preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=-1
        )),
    ]),
    'Bagging Regressor': Pipeline([
        ('prep', tree_preprocessor),
        ('model', BaggingRegressor(
            n_estimators=20,
            random_state=42,
            n_jobs=-1
        )),
    ]),
    'ADA Boost Regressor': Pipeline([
        ('prep', tree_preprocessor),
        ('model', AdaBoostRegressor(
            n_estimators=80,
            learning_rate=0.2,
            random_state=42
        )),
    ]),
    'Gradient Boosting Regressor': Pipeline([
        ('prep', tree_preprocessor),
        ('model', GradientBoostingRegressor(
            learning_rate=0.05,
            n_estimators=150,
            max_depth=5,
            subsample=0.8,
            random_state=42
        )),
    ]),
}

In [56]:
# -----------------------------
# 7) Baseline/test comparison for 9 models
# -----------------------------
print('\n' + '=' * 90)
print('Running 9-model comparison')
print('=' * 90)

comparison_rows = []
fitted_test_models = {}
test_predictions = {}

for name, pipe in models.items():
    fitted_model, y_pred, metrics = evaluate_test(pipe, X_train, y_train, X_test, y_test)
    fitted_test_models[name] = fitted_model
    test_predictions[name] = y_pred
    comparison_rows.append({
        'Model': name,
        'MSE': metrics['MSE'],
        'RMSE': metrics['RMSE'],
        'MAE': metrics['MAE'],
        'R2': metrics['R2'],
    })
    print(f"{name:<32} RMSE={metrics['RMSE']:.4f} | R2={metrics['R2']:.4f}")

comparison_df = pd.DataFrame(comparison_rows).sort_values('RMSE').reset_index(drop=True)
save_table(comparison_df, '23_comparison_of_performance_metrics.csv')


Running 9-model comparison
Multiple Linear Regression       RMSE=507.5323 | R2=0.7180
Ridge Regression                 RMSE=506.2222 | R2=0.7195
LASSO Regression                 RMSE=505.7242 | R2=0.7200
Decision Tree Regression         RMSE=457.8522 | R2=0.7705
Support Vector Regression        RMSE=751.4119 | R2=0.3819
Random Forest Regression         RMSE=404.1031 | R2=0.8212
Bagging Regressor                RMSE=417.6386 | R2=0.8091
ADA Boost Regressor              RMSE=523.6863 | R2=0.6998
Gradient Boosting Regressor      RMSE=406.4756 | R2=0.8191


WindowsPath('paper_reproduction_outputs_revised2/23_comparison_of_performance_metrics.csv')

In [57]:
# -----------------------------
# 8) Detailed comparison set: SVR, RF, GBR
# -----------------------------
svr_model = models['Support Vector Regression']
rf_model = models['Random Forest Regression']
gbr_default_model = models['Gradient Boosting Regressor']

# GBR grid search according to paper
param_grid = {
    'model__learning_rate': [0.05, 0.10],
    'model__max_depth': [3, 5],
    'model__n_estimators': [100, 150],
    'model__subsample': [0.8, 1.0],
}

grid = GridSearchCV(
    estimator=Pipeline([
        ('prep', tree_preprocessor),
        ('model', GradientBoostingRegressor(random_state=42)),
    ]),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)
grid.fit(X_train, y_train)

best_gbr = grid.best_estimator_
best_params = grid.best_params_
best_cv_rmse = -grid.best_score_

best_gbr_pred = best_gbr.predict(X_test)
best_gbr_metrics = regression_metrics(y_test, best_gbr_pred)

best_summary = pd.DataFrame([{
    'Best_learning_rate': best_params['model__learning_rate'],
    'Best_max_depth': best_params['model__max_depth'],
    'Best_n_estimators': best_params['model__n_estimators'],
    'Best_subsample': best_params['model__subsample'],
    'GridSearch_CV_RMSE': best_cv_rmse,
    'Test_MSE': best_gbr_metrics['MSE'],
    'Test_RMSE': best_gbr_metrics['RMSE'],
    'Test_MAE': best_gbr_metrics['MAE'],
    'Test_R2': best_gbr_metrics['R2'],
}])
save_table(best_summary, '02_best_gbr_summary.csv')
pd.DataFrame(grid.cv_results_).to_csv(OUT_DIR / '02_gbr_gridsearch_full_results.csv', index=False)

print('\nBest tuned GBR parameters:')
print(best_params)
print('Best tuned GBR metrics on test set:')
for k, v in best_gbr_metrics.items():
    print(f'  {k}: {v:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best tuned GBR parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 150, 'model__subsample': 0.8}
Best tuned GBR metrics on test set:
  MSE: 165222.4457
  RMSE: 406.4756
  MAE: 261.2475
  R2: 0.8191


In [59]:
# -----------------------------
# 9) Figure 1 - CV RMSE line plot for tuned GBR
# -----------------------------
gbr_fold_rmse = []
for train_idx, valid_idx in cv.split(X_train, y_train):
    X_tr = X_train.iloc[train_idx]
    X_va = X_train.iloc[valid_idx]
    y_tr = y_train.iloc[train_idx]
    y_va = y_train.iloc[valid_idx]
    est = clone(best_gbr)
    est.fit(X_tr, y_tr)
    pred = est.predict(X_va)
    gbr_fold_rmse.append(np.sqrt(mean_squared_error(y_va, pred)))

fold_positions = np.arange(1, 6)
fold_rmse_df = pd.DataFrame({'Fold': fold_positions, 'RMSE': gbr_fold_rmse})
save_table(fold_rmse_df, '01_cross_validation_rmse_scores_best_tuned_gbr.csv')

plt.figure(figsize=(8, 5))
plt.plot(fold_positions, gbr_fold_rmse, marker='o')
plt.title('Cross-Validation RMSE Scores')
plt.xlabel('Fold')
plt.ylabel('RMSE')
plt.xticks(np.arange(1, 5.1, 0.5))
plt.tight_layout()
plt.savefig(OUT_DIR / '01_cross_validation_rmse_scores_best_tuned_gbr.png', dpi=200)
plt.close()

In [60]:
# -----------------------------
# 10) Figure 2 - Grid search RMSE heatmap
# -----------------------------
grid_results = pd.DataFrame(grid.cv_results_).copy()
grid_results['mean_RMSE'] = -grid_results['mean_test_score']
heat = (
    grid_results
    .groupby(['param_model__max_depth', 'param_model__learning_rate'])['mean_RMSE']
    .mean()
    .unstack()
    .sort_index()
)
heat.to_csv(OUT_DIR / '02_grid_search_rmse_heatmap_values.csv')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(heat.values, aspect='auto', cmap='coolwarm')
ax.set_title('Grid Search RMSE Heatmap')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Max Depth')
ax.set_xticks(np.arange(len(heat.columns)))
ax.set_yticks(np.arange(len(heat.index)))
ax.set_xticklabels([str(v) for v in heat.columns])
ax.set_yticklabels([str(v) for v in heat.index])
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f'{heat.values[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im, ax=ax, label='RMSE')
plt.tight_layout()
plt.savefig(OUT_DIR / '02_grid_search_rmse_heatmap.png', dpi=200)
plt.close()

In [61]:
# -----------------------------
# 11) Figure 3 - Top 10 permutation importance
# -----------------------------
perm = permutation_importance(
    best_gbr,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring='r2',
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'Mean Importance': perm.importances_mean,
    'Std Importance': perm.importances_std,
}).sort_values('Mean Importance', ascending=False)
importance_top10 = importance_df.head(10).copy()
save_table(importance_top10, '03_top10_permutation_feature_importances.csv')

plt.figure(figsize=(10, 6))
plt.barh(importance_top10['feature'].iloc[::-1], importance_top10['Mean Importance'].iloc[::-1])
plt.title('Top 10 Permutation Feature importances')
plt.xlabel('Mean Importance')
plt.ylabel('')
plt.tight_layout()
plt.savefig(OUT_DIR / '03_top10_permutation_feature_importances.png', dpi=200)
plt.close()

In [62]:
# -----------------------------
# 12) Figure/Table 4/5 - RMSE across folds for GBR, RF, SVR
# -----------------------------
detailed_models = {
    'GBR': best_gbr,
    'RF': rf_model,
    'SVR': svr_model,
}
fold_compare_df = build_fold_rmse_table(detailed_models, X_train, y_train, cv)
save_table(fold_compare_df, '04_rmse_comparison_across_5_folds_gbr_rf_svr.csv')

plot_df = fold_compare_df[fold_compare_df['Fold'] != 'Mean'].copy()
plot_df_long = plot_df.melt(id_vars='Fold', var_name='Model', value_name='RMSE')

fig, ax = plt.subplots(figsize=(10, 6))
models_order = ['GBR', 'RF', 'SVR']
fold_labels = plot_df['Fold'].tolist()
x = np.arange(len(fold_labels))
width = 0.25
for i, model_name in enumerate(models_order):
    ax.bar(x + (i - 1) * width, plot_df[model_name].values, width=width, label=model_name)
ax.set_title('RMSE Comparision Across Folds')
ax.set_xlabel('Fold')
ax.set_ylabel('RMSE')
ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.legend(title='Model')
plt.tight_layout()
plt.savefig(OUT_DIR / '05_barplot_rmse_across_folds_gbr_rf_svr.png', dpi=200)
plt.close()

In [63]:
# -----------------------------
# 13) ML-only plots (MLR/Ridge/LASSO/DTR/SVR)
# -----------------------------
ml_names = [
    'Multiple Linear Regression',
    'Ridge Regression',
    'LASSO Regression',
    'Decision Tree Regression',
    'Support Vector Regression',
]
ml_df = comparison_df[comparison_df['Model'].isin(ml_names)].copy()

# 6. Prediction error for multiple linear regression
mlr_actual = y_test.values
mlr_pred = test_predictions['Multiple Linear Regression']
mlr_r2 = regression_metrics(y_test, mlr_pred)['R2']
coef = fit_line(mlr_actual, mlr_pred)
line_x = np.linspace(mlr_actual.min(), mlr_actual.max(), 200)
line_y = coef[0] * line_x + coef[1]

plt.figure(figsize=(6, 6))
plt.scatter(mlr_actual, mlr_pred, alpha=0.5, label=f'$R^2$ = {mlr_r2:.3f}')
plt.plot(line_x, line_y, linestyle='--', linewidth=2, label='Best fit')
plt.plot(line_x, line_x, linestyle='--', linewidth=1, label='Identity')
plt.xlim(-1000, 3000)
plt.title('Prediction Error for Linear Regression')
plt.xlabel('y')
plt.ylabel('y hat')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(OUT_DIR / '06_prediction_error_linear_regression.png', dpi=200)
plt.close()

# 7. Ridge actual vs predicted
ridge_pred = test_predictions['Ridge Regression']
ridge_r2 = regression_metrics(y_test, ridge_pred)['R2']
plt.figure(figsize=(6, 6))
plt.scatter(y_test, ridge_pred, alpha=0.5)
min_v = min(np.min(y_test), np.min(ridge_pred))
max_v = max(np.max(y_test), np.max(ridge_pred))
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--', linewidth=1)
plt.xlim(0, 3000)
plt.title(f'R-squared (R^2): {ridge_r2:.3f}')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.tight_layout()
plt.savefig(OUT_DIR / '07_prediction_error_ridge_regression.png', dpi=200)
plt.close()

# 8. LASSO actual vs predicted
lasso_pred = test_predictions['LASSO Regression']
lasso_r2 = regression_metrics(y_test, lasso_pred)['R2']
plt.figure(figsize=(6, 6))
plt.scatter(y_test, lasso_pred, alpha=0.5)
min_v = min(np.min(y_test), np.min(lasso_pred))
max_v = max(np.max(y_test), np.max(lasso_pred))
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--', linewidth=1)
plt.xlim(0, 3000)
plt.title(f'R-squared (R^2): {lasso_r2:.3f}')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.tight_layout()
plt.savefig(OUT_DIR / '08_prediction_error_lasso_regression.png', dpi=200)
plt.close()

In [64]:
# 9. Decision tree hyperparameter line plot (combination sweep)
dtr_combinations = list(product(
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,  None],   # max_depth (20)
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],           # min_samples_leaf(19)
))

dtr_rows = []

for idx, (depth, min_leaf) in enumerate(dtr_combinations, start=1):
    model = Pipeline([
        ('prep', tree_preprocessor),
        ('model', DecisionTreeRegressor(
            max_depth=depth,
            min_samples_split=2,
            min_samples_leaf=min_leaf,
            random_state=42
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    dtr_rows.append({
        'Hyperparameters': idx,
        'max_depth': depth,
        'min_samples_split': 2,
        'min_samples_leaf': min_leaf,
        'R2': m['R2']
    })

dtr_df = pd.DataFrame(dtr_rows)
save_table(dtr_df, '09_decision_tree_r2_hyperparameter_curve.csv')

plt.figure(figsize=(8, 5))
plt.plot(dtr_df['Hyperparameters'], dtr_df['R2'])
plt.title(f'R-squared for Decision Tree Regression R-Squared: {dtr_df["R2"].max():.3f}')
plt.xlabel('Hyperparameters')
plt.ylabel('R-Squared')
plt.xticks(np.arange(0, len(dtr_df) + 1, 50))
plt.tight_layout()
plt.savefig(OUT_DIR / '09_decision_tree_r2_hyperparameter_curve.png', dpi=200)
plt.close()

In [65]:
# -----------------------------
# 10. SVR hyperparameter line plot (3000 combinations)
# -----------------------------

# 25 × 20 × 6 = 3000
C_list = np.logspace(-2, 4, 25)                 # 0.01 ~ 10000
gamma_list = np.logspace(-4, 1, 20)             # 0.0001 ~ 10
epsilon_list = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5]

svr_combinations = list(product(C_list, gamma_list, epsilon_list))

svr_rows = []

for idx, (c, gamma, eps) in enumerate(svr_combinations, start=1):
    model = Pipeline([
        ('prep', scaled_preprocessor),
        ('model', SVR(
            kernel='rbf',
            C=float(c),
            gamma=float(gamma),
            epsilon=float(eps)
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    svr_rows.append({
        'Combination_Index': idx,
        'C': float(c),
        'gamma': float(gamma),
        'epsilon': float(eps),
        'R2': m['R2']
    })

svr_df = pd.DataFrame(svr_rows)
save_table(svr_df, '10_svr_r2_hyperparameter_curve.csv')

plt.figure(figsize=(8, 5))

plt.scatter(
    svr_df['Combination_Index'],
    svr_df['R2'],
    alpha=0.5   # 위 그래프들과 동일
)

plt.title(f'SVR with RBF Kernel R-squared: {svr_df["R2"].max():.3f}')
plt.xlabel('Hyperparameter Combinations')
plt.ylabel('R^2')

plt.xticks(np.arange(0, 3001, 500))
plt.tight_layout()
plt.savefig(OUT_DIR / '10_svr_r2_hyperparameter_curve.png', dpi=200)
plt.close()

In [66]:
# SVR actual vs predicted
svr_pred = test_predictions['Support Vector Regression']
svr_r2 = regression_metrics(y_test, svr_pred)['R2']

plt.figure(figsize=(6, 6))

# 🔥 점 그래프 (스타일 통일)
plt.scatter(y_test, svr_pred, s=20, alpha=0.5)

# 🔥 y = x 기준선
min_v = min(np.min(y_test), np.min(svr_pred))
max_v = max(np.max(y_test), np.max(svr_pred))
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--', linewidth=1)

plt.xlim(0, 3000)   # 기존 스타일 유지

plt.title(f'R-squared (R^2): {svr_r2:.3f}')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')

plt.tight_layout()
plt.savefig(OUT_DIR / '10_prediction_error_svr.png', dpi=200)
plt.close()

In [91]:
# 11-14. Comparison bar plots for ML algorithms
ml_short = ['MLR', 'RR', 'LR', 'SVR', 'DTR']
ml_metric_source = pd.DataFrame({
    'Model': ml_short,
    'R2': [
        comparison_df.loc[comparison_df['Model'] == 'Multiple Linear Regression', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Ridge Regression', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'LASSO Regression', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Support Vector Regression', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Decision Tree Regression', 'R2'].iloc[0],
    ],
    'MSE': [
        comparison_df.loc[comparison_df['Model'] == 'Multiple Linear Regression', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Ridge Regression', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'LASSO Regression', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Support Vector Regression', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Decision Tree Regression', 'MSE'].iloc[0],
    ],
    'RMSE': [
        comparison_df.loc[comparison_df['Model'] == 'Multiple Linear Regression', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Ridge Regression', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'LASSO Regression', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Support Vector Regression', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Decision Tree Regression', 'RMSE'].iloc[0],
    ],
    'MAE': [
        comparison_df.loc[comparison_df['Model'] == 'Multiple Linear Regression', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Ridge Regression', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'LASSO Regression', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Support Vector Regression', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Decision Tree Regression', 'MAE'].iloc[0],
    ],
})
save_table(ml_metric_source, '11_14_ml_algorithm_metric_summary.csv')

def plot_bar(x, y, title, xlabel, ylabel, filename):
    plt.figure(figsize=(8, 5))

    bars = plt.bar(x, y, width=0.5)

    for i, v in enumerate(y):
        plt.text(
            i, v, f'{v:.2f}', #i, v + max(y)*0.01, f'{v:.2f}',
            ha='center', 
            va='bottom', 
            fontsize=13
        )

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=200)
    plt.close()

plot_bar(ml_metric_source['Model'], ml_metric_source['R2'], 'Error States Comparision', 'R^2', 'Range of Error', '11_comparison_r_squared_ml_algorithms.png')
plot_bar(ml_metric_source['Model'], ml_metric_source['MSE'], 'Comparision MSE', 'MSE', 'Range of MSE', '12_comparison_mse_ml_algorithms.png')
plot_bar(ml_metric_source['Model'], ml_metric_source['RMSE'], 'Comparision RMSE', 'RMSE', 'Range of RMSE', '13_comparison_rmse_ml_algorithms.png')
plot_bar(ml_metric_source['Model'], ml_metric_source['MAE'], 'Comparision MAE', 'MAE', 'Range of MSE', '14_comparison_mae_ml_algorithms.png')

In [93]:
# -----------------------------
# 14) Ensemble hyperparameter sweeps and ensemble comparisons
# -----------------------------
# 15. Random Forest regression performance lines
rf_param_grid = list(product(
    [8, 10, 12, None],       # max_depth
    [1, 2, 3, 4],              # min_samples_leaf
    [100, 150, 200],     # n_estimators
))

rf_sweep_rows = []

for idx, (max_d, min_leaf, n_est) in enumerate(rf_param_grid, start=1):
    model = Pipeline([
        ('prep', tree_preprocessor),   # 여기에 StandardScaler 포함되어 있다고 가정
        ('model', RandomForestRegressor(
            n_estimators=n_est,
            max_depth=max_d,
            min_samples_leaf=min_leaf,
            min_samples_split=2,
            random_state=42,
            n_jobs=-1
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    rf_sweep_rows.append({
        'Combination_Index': idx,
        'n_estimators': n_est,
        'max_depth': max_d if max_d is not None else 'None',
        'min_samples_leaf': min_leaf,
        **m
    })

rf_sweep_df = pd.DataFrame(rf_sweep_rows)

# CSV 저장
save_table(rf_sweep_df, '15_random_forest_performance_combinations.csv')

# 그래프 저장
for metric, fname, title_prefix in [
    ('R2',   '15a_random_forest_r2.png',   'R-squared for Random Forest Regression\nR-squared:'),
    ('MSE',  '15b_random_forest_mse.png',  'MSE for Random Forest Regression\nMSE:'),
    ('RMSE', '15c_random_forest_rmse.png', 'RMSE for Random Forest Regression\nRMSE:'),
    ('MAE',  '15d_random_forest_mae.png',  'MAE for Random Forest Regression\nMAE:'),
]:
    plt.figure(figsize=(8, 5))
    plt.plot(rf_sweep_df['Combination_Index'], rf_sweep_df[metric])

    if metric == 'R2':
        best_value = rf_sweep_df[metric].max()
        best_idx = rf_sweep_df[metric].idxmax()
    else:
        best_value = rf_sweep_df[metric].min()
        best_idx = rf_sweep_df[metric].idxmin()

    plt.title(f"{title_prefix} {best_value:.3f}")
    plt.xlabel('Hyperparameter Combinations')
    plt.ylabel(metric if metric != 'R2' else 'R-squared')
    plt.xticks(np.arange(0, len(rf_sweep_df) + 1, 10))
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.close()

# 최적 조합 확인용 출력
print("Best R2 combination:")
print(rf_sweep_df.loc[rf_sweep_df['R2'].idxmax()])

print("\nBest MSE/RMSE/MAE-side combination (by RMSE):")
print(rf_sweep_df.loc[rf_sweep_df['RMSE'].idxmin()])

Best R2 combination:
Combination_Index               39
n_estimators                   200
max_depth                     None
min_samples_leaf                 1
MSE                  163299.314637
RMSE                    404.103099
MAE                      253.77662
R2                        0.821231
Name: 38, dtype: object

Best MSE/RMSE/MAE-side combination (by RMSE):
Combination_Index               39
n_estimators                   200
max_depth                     None
min_samples_leaf                 1
MSE                  163299.314637
RMSE                    404.103099
MAE                      253.77662
R2                        0.821231
Name: 38, dtype: object


In [94]:
# 16. Bagging regressor performance lines
# x축은 그대로 1~20
shown_values = np.arange(1, 21)

# 실제 n_estimators는 홀수로 증가 (20개)
actual_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

bag_rows = []

for shown_v, actual_v in zip(shown_values, actual_values):
    model = Pipeline([
        ('prep', tree_preprocessor),
        ('model', BaggingRegressor(
            n_estimators=actual_v,
            random_state=42,
            n_jobs=-1
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    bag_rows.append({
        'Hyperparameters': shown_v,   # 그래프용 (1~20)
        'n_estimators': actual_v,     # 실제 값 (1,3,5,...)
        **m
    })

bag_df = pd.DataFrame(bag_rows)
save_table(bag_df, '16_bagging_performance_sweep.csv')

for metric, fname, title_prefix in [
    ('R2', '16a_bagging_r2.png', 'R-squared for Bagging Regressor R-squared:'),
    ('MSE', '16b_bagging_mse.png', 'MSE for Bagging Regressor MSE:'),
    ('RMSE', '16c_bagging_rmse.png', 'RMSE for Bagging Regressor RMSE:'),
    ('MAE', '16d_bagging_mae.png', 'MAE for Bagging Regressor MAE:'),
]:
    plt.figure(figsize=(8, 5))
    plt.plot(bag_df['Hyperparameters'], bag_df[metric])
    suffix = f" {bag_df[metric].max():.3f}" if metric == 'R2' else f" {bag_df[metric].min():.3f}"
    plt.title(title_prefix + suffix)
    plt.xlabel('Hyperparameters')
    plt.ylabel(metric if metric != 'R2' else 'R^2')
    plt.xticks(np.arange(0, 20.1, 2.5))
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.close()

In [95]:
# 17. AdaBoost regressor performance lines
n_estimators_list = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190]   # 20개
learning_rate_list = [0.01, 0.05, 0.1, 0.2]               # 4개

ada_combinations = list(product(n_estimators_list, learning_rate_list))

ada_rows = []

for idx, (n_est, lr) in enumerate(ada_combinations, start=1):
    model = Pipeline([
        ('prep', tree_preprocessor),
        ('model', AdaBoostRegressor(
            n_estimators=n_est,
            learning_rate=lr,
            random_state=42
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    ada_rows.append({
        'Combination_Index': idx,
        'n_estimators': n_est,
        'learning_rate': lr,
        **m
    })

ada_df = pd.DataFrame(ada_rows)
save_table(ada_df, '17_adaboost_performance_combinations.csv')

for metric, fname, title_prefix in [
    ('R2', '17a_adaboost_r2.png', 'R-squared for AdaBoostRegressor Max R-squared:'),
    ('MSE', '17b_adaboost_mse.png', 'MSE for AdaBoostRegressor Min MSE:'),
    ('RMSE', '17c_adaboost_rmse.png', 'RMSE for AdaBoostRegressor Min RMSE:'),
    ('MAE', '17d_adaboost_mae.png', 'MAE for AdaBoostRegressor Min MAE:'),
]:
    plt.figure(figsize=(8, 5))
    plt.plot(ada_df['Combination_Index'], ada_df[metric])

    suffix = f" {ada_df[metric].max():.3f}" if metric == 'R2' else f" {ada_df[metric].min():.3f}"

    plt.title(title_prefix + suffix)
    plt.xlabel('Hyperparameter Combinations')
    plt.ylabel(metric if metric != 'R2' else 'R^2')

    plt.xticks(np.arange(0, 81, 10))
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.close()

In [96]:
# 18. Gradient Boosting regressor performance lines
n_estimators_list = [50, 100, 150]   # 3
learning_rate_list = [0.01, 0.05, 0.1]      # 3
max_depth_list = [3, 5, 7]              # 3

gbr_combinations = list(product(
    learning_rate_list,
    n_estimators_list,
    max_depth_list
))

gbr_rows = []

for idx, (lr, n_est, depth) in enumerate(gbr_combinations, start=1):
    model = Pipeline([
        ('prep', tree_preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=depth,
            subsample=1.0,
            random_state=42,
        )),
    ])

    _, pred, m = evaluate_test(model, X_train, y_train, X_test, y_test)

    gbr_rows.append({
        'Combination_Index': idx,   # 🔥 핵심
        'n_estimators': n_est,
        'learning_rate': lr,
        'max_depth': depth,
        **m
    })

gbr_df = pd.DataFrame(gbr_rows)
save_table(gbr_df, '18_gradient_boosting_performance_combinations.csv')

for metric, fname, title_prefix in [
    ('R2', '18a_gbr_r2.png', 'R-squared for Gradient Boosting Regressor Max R-squared:'),
    ('MSE', '18b_gbr_mse.png', 'MSE for Gradient Boosting Regressor Min MSE:'),
    ('RMSE', '18c_gbr_rmse.png', 'RMSE for Gradient Boosting Regressor Min RMSE:'),
    ('MAE', '18d_gbr_mae.png', 'MAE for Gradient Boosting Regressor Min MAE:'),
]:
    plt.figure(figsize=(8, 5))
    plt.plot(gbr_df['Combination_Index'], gbr_df[metric])

    suffix = f" {gbr_df[metric].max():.3f}" if metric == 'R2' else f" {gbr_df[metric].min():.3f}"

    plt.title(title_prefix + suffix)
    plt.xlabel('Hyperparameter Combinations')   # 🔥 변경
    plt.ylabel(metric if metric != 'R2' else 'R^2')

    plt.xticks(np.arange(0, len(gbr_df)+1, 5))
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200)
    plt.close()

In [97]:
# 19-22 Ensemble comparison bars
ensemble_metric_source = pd.DataFrame({
    'Model': ['RFR', 'BR', 'ADA BR', 'GBR'],
    'R2': [
        comparison_df.loc[comparison_df['Model'] == 'Random Forest Regression', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Bagging Regressor', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'ADA Boost Regressor', 'R2'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Gradient Boosting Regressor', 'R2'].iloc[0],
    ],
    'MSE': [
        comparison_df.loc[comparison_df['Model'] == 'Random Forest Regression', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Bagging Regressor', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'ADA Boost Regressor', 'MSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Gradient Boosting Regressor', 'MSE'].iloc[0],
    ],
    'RMSE': [
        comparison_df.loc[comparison_df['Model'] == 'Random Forest Regression', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Bagging Regressor', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'ADA Boost Regressor', 'RMSE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Gradient Boosting Regressor', 'RMSE'].iloc[0],
    ],
    'MAE': [
        comparison_df.loc[comparison_df['Model'] == 'Random Forest Regression', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Bagging Regressor', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'ADA Boost Regressor', 'MAE'].iloc[0],
        comparison_df.loc[comparison_df['Model'] == 'Gradient Boosting Regressor', 'MAE'].iloc[0],
    ],
})
save_table(ensemble_metric_source, '19_22_ensemble_metric_summary.csv')

def plot_bar(x, y, title, xlabel, ylabel, filename):
    plt.figure(figsize=(8, 5))

    bars = plt.bar(x, y, width=0.5)

    for i, v in enumerate(y):
        plt.text(
            i, v, f'{v:.2f}', #i, v + max(y)*0.01, f'{v:.2f}',
            ha='center', 
            va='bottom', 
            fontsize=13
        )

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=200)
    plt.close()
    
plot_bar(ensemble_metric_source['Model'], ensemble_metric_source['R2'], 'Error States Comparision', 'R^2', 'Range of Error', '19_comparison_r_squared_ensemble_models.png')
plot_bar(ensemble_metric_source['Model'], ensemble_metric_source['MSE'], 'Comparsion MSE', 'MSE', 'Range of MSE', '20_comparison_mse_ensemble_models.png')
plot_bar(ensemble_metric_source['Model'], ensemble_metric_source['RMSE'], 'Comparision RMSE', 'RMSE', 'Range of RMSE', '21_comparison_rmse_ensemble_models.png')
plot_bar(ensemble_metric_source['Model'], ensemble_metric_source['MAE'], 'Comparision MAE', 'MAE', 'Range of MAE', '22_comparison_mae_ensemble_models.png')

In [98]:
# -----------------------------
# 15) Summary files
# -----------------------------
summary_lines = [
    'Revised paper-style reproduction summary',
    '=' * 60,
    f'Dataset shape: {df.shape}',
    f'Train shape: {X_train.shape}',
    f'Test shape: {X_test.shape}',
    '',
    '9-model test-set comparison:',
]
for _, row in comparison_df.iterrows():
    summary_lines.append(
        f"- {row['Model']}: RMSE={row['RMSE']:.4f}, MSE={row['MSE']:.4f}, MAE={row['MAE']:.4f}, R2={row['R2']:.4f}"
    )
summary_lines += [
    '',
    'Best tuned GBR parameters:',
    str(best_params),
    f"Best tuned GBR CV RMSE: {best_cv_rmse:.4f}",
    f"Best tuned GBR Test RMSE: {best_gbr_metrics['RMSE']:.4f}",
    f"Best tuned GBR Test MSE: {best_gbr_metrics['MSE']:.4f}",
    f"Best tuned GBR Test MAE: {best_gbr_metrics['MAE']:.4f}",
    f"Best tuned GBR Test R2: {best_gbr_metrics['R2']:.4f}",
    '',
    '5-fold RMSE comparison table (GBR/RF/SVR):',
    fold_compare_df.to_string(index=False),
    '',
    'Paired t-test on fold RMSE:',
    ttest_df.to_string(index=False),
    '',
    'Top 10 permutation importances:',
]
for _, row in importance_top10.iterrows():
    summary_lines.append(f"- {row['feature']}: {row['Mean Importance']:.6f}")

(OUT_DIR / 'summary.txt').write_text('\n'.join(summary_lines), encoding='utf-8')

# Also store the paper numbers user explicitly mentioned
paper_reference_metrics = pd.DataFrame([
    {
        'Scenario': 'Paper abstract / Table 6 style numbers',
        'Model': 'GBR',
        'learning_rate': 0.1,
        'n_estimators': 100,
        'max_depth': 3,
        'R2': 0.83,
        'MSE': 158559.33,
        'RMSE': 399.44,
        'MAE': 253.62,
    }
])
save_table(paper_reference_metrics, '24_paper_reported_gbr_reference_metrics.csv')

print('\n' + '=' * 90)
print('Finished.')
print(f'Outputs saved to: {OUT_DIR.resolve()}')
print('=' * 90)

NameError: name 'ttest_df' is not defined